# Reproducing EquiCity's headline findings

Every number in the research paper, recomputed from the aggregated tables in `processed_data/analytics/` and the income series in `raw_data/`. To rebuild those tables from the raw transactions, run `bash reproduce.sh`.

In [1]:
import os, sys
import numpy as np, pandas as pd
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'dashboard'))
from analysis import affordability as A, spatial as S
T = lambda n: pd.read_csv(os.path.join(ROOT, 'processed_data', 'analytics', n))
pd.set_option('display.width', 140)

## 1. The index tracks HDB's official Resale Price Index

In [2]:
from analysis.temporal import national_quarterly_vs_official
v = national_quarterly_vs_official().dropna()
d = np.corrcoef(np.diff(np.log(v.equicity)), np.diff(np.log(v.official_rpi.astype(float))))[0, 1]
print(f'quarterly-change correlation {d:.3f}; latest quarter {v.quarter.iloc[-1]}: '
      f'EquiCity {v.equicity.iloc[-1]:.1f} vs official {float(v.official_rpi.iloc[-1]):.1f}')

quarterly-change correlation 0.834; latest quarter 2026-Q2: EquiCity 208.4 vs official 202.8


## 2. Affordability of the national constant-quality 4-room flat (HDB loan, 30% MSR)

In [3]:
t = T('hedonic_town_year_type.csv'); t4 = t[(t.flat_type == '4-Room') & (t.n_town >= 30)]
w = t4[t4.year.between(2015, 2025)].groupby('town').n_town.sum(); w /= w.sum()
nat = t4.assign(w=t4.town.map(w).fillna(0)).groupby('year').apply(
    lambda d: np.exp((d.log_price_ref * d.w).sum() / d.w.sum()), include_groups=False)
rows = []
for y in [2000, 2005, 2010, 2013, 2015, 2019, 2023, 2025, 2026]:
    e50, e20 = A.evaluate(nat[y], y, 50), A.evaluate(nat[y], y, 20)
    rows.append((y, round(nat[y]), round(e50['msr_pct'], 1), round(e20['msr_pct'], 1),
                 round(e50['required_percentile']), round(e50['price_to_annual_income'], 2)))
pd.DataFrame(rows, columns=['year', 'price', 'MSR at P50 %', 'MSR at P20 %', 'required pct', 'price/income'])

,year,price,MSR at P50 %,MSR at P20 %,required pct,price/income
0,2000,229202,19.5,39.0,30,4.78
1,2005,208886,16.4,35.5,25,4.01
2,2010,342898,20.5,44.9,34,5.01
3,2013,437098,21.2,48.7,36,5.20
4,2015,391384,17.5,39.2,28,4.28
5,2019,378289,15.4,35.8,25,3.78
6,2023,525272,18.5,42.9,30,4.54
7,2025,613722,19.7,45.1,33,4.83
8,2026,612751,19.7,45.0,32,4.82


In [4]:
af30 = A.affordability_frame(t4.rename(columns={'price_ref': 'price'}), 30)
af30.groupby('year').within_cap.sum().loc[[2013, 2015, 2019, 2022, 2025, 2026]]

year
2013     2
2015    13
2019    16
2022    12
2025     9
2026     9
Name: within_cap, dtype: int64

## 3. Spatial clustering of location premia

In [5]:
rs = T('robust_moran_summary.csv')
print(f"Rising 1995-99 -> 2022-26: {(rs.change > 0).sum()} of {len(rs)} specifications")
print(f"PRIMARY  significant after Hamed-Rao serial-correlation correction: "
      f"{((rs.mk_hamed_rao_p < 0.05) & (rs.change > 0)).sum()} of {len(rs)}")
print(f"(unadjusted Mann-Kendall, which assumes independent years and overstates confidence: "
      f"{((rs.mann_kendall_p < 0.05) & (rs.mann_kendall_z > 0)).sum()} of {len(rs)})")
rs[['spec', 'I_1995_99', 'I_2022_26', 'change', 'lag1_acf_detrended', 'mk_hamed_rao_p', 'mann_kendall_p']].round(3)

Rising 1995-99 -> 2022-26: 25 of 25 specifications
PRIMARY  significant after Hamed-Rao serial-correlation correction: 24 of 25
(unadjusted Mann-Kendall, which assumes independent years and overstates confidence: 25 of 25)


,spec,I_1995_99,I_2022_26,change,lag1_acf_detrended,mk_hamed_rao_p,mann_kendall_p
0,Distance band 4 km | binary,0.547,0.743,0.197,0.202,0.000,0.0
1,Distance band 4 km | row-standardised,0.560,0.774,0.214,0.510,0.000,0.0
2,Distance band 6 km | binary,0.474,0.676,0.202,0.417,0.000,0.0
3,Distance band 6 km | row-standardised,0.573,0.728,0.156,0.373,0.000,0.0
4,Distance band 8 km | binary,0.392,0.497,0.105,0.458,0.000,0.0
5,Distance band 8 km | row-standardised,0.477,0.549,0.072,0.257,0.000,0.0
6,Inverse distance | raw,0.151,0.180,0.029,0.265,0.000,0.0
7,Inverse distance | row-standardised,0.153,0.182,0.029,0.130,0.000,0.0
8,kNN k=10 | binary,0.275,0.342,0.067,0.361,0.000,0.0
9,kNN k=10 | row-standardised,0.275,0.342,0.067,0.361,0.000,0.0


Thirty-two yearly values are few for estimating serial dependence. `trend_sensitivity.py` checks the Hamed-Rao result against its lag choice and simulates how often persistence alone (trendless AR(1) series) produces a rise as consistent as the observed one.

In [6]:
ts = T('robust_trend_sensitivity.csv')
for c in ['mk_hamed_rao_p_all_lags', 'mk_hamed_rao_p_lags_1_3', 'sim_p_fitted_phi', 'sim_p_ar2',
          'sim_p_phi_uncertain', 'sim_p_phi_0_9']:
    print(f'{c:26s} {(ts[c] < 0.05).sum()} of {len(ts)} below 0.05')
ts.round(4)

mk_hamed_rao_p_all_lags    24 of 25 below 0.05
mk_hamed_rao_p_lags_1_3    24 of 25 below 0.05
sim_p_fitted_phi           24 of 25 below 0.05
sim_p_ar2                  24 of 25 below 0.05
sim_p_phi_uncertain        24 of 25 below 0.05
sim_p_phi_0_9              23 of 25 below 0.05


,spec,phi_ar1_detrended,phi_ar1_bias_corrected,ar2_phi1,ar2_phi2,mk_original_p,mk_hamed_rao_p_all_lags,mk_hamed_rao_p_lags_1_3,mk_tfpw_p,sim_p_fitted_phi,sim_p_phi_0_9,sim_p_ar2,sim_p_phi_uncertain
0,Distance band 4 km | binary,0.2015,0.2516,0.2388,-0.1853,0.0000,0.0000,0.0000,0.0,0.0002,0.0002,0.0002,0.0002
1,Distance band 4 km | row-standardised,0.5102,0.5892,0.5702,-0.1177,0.0000,0.0000,0.0000,0.0,0.0002,0.0002,0.0002,0.0002
2,Distance band 6 km | binary,0.4174,0.4877,0.4893,-0.1723,0.0000,0.0000,0.0000,0.0,0.0002,0.0002,0.0002,0.0002
3,Distance band 6 km | row-standardised,0.3726,0.4387,0.5063,-0.3590,0.0000,0.0000,0.0000,0.0,0.0002,0.0002,0.0002,0.0002
4,Distance band 8 km | binary,0.4576,0.5318,0.5024,-0.0978,0.0000,0.0000,0.0000,0.0,0.0002,0.0070,0.0002,0.0002
5,Distance band 8 km | row-standardised,0.2567,0.3120,0.2741,-0.0677,0.0000,0.0000,0.0000,0.0,0.0002,0.0225,0.0002,0.0002
6,Inverse distance | raw,0.2653,0.3215,0.2575,0.0297,0.0000,0.0001,0.0003,0.0,0.0002,0.0545,0.0002,0.0007
7,Inverse distance | row-standardised,0.1296,0.1730,0.1377,-0.0630,0.0000,0.0000,0.0000,0.0,0.0002,0.0317,0.0002,0.0002
8,kNN k=10 | binary,0.3606,0.4257,0.3783,-0.0489,0.0000,0.0000,0.0000,0.0,0.0002,0.0130,0.0002,0.0002
9,kNN k=10 | row-standardised,0.3606,0.4257,0.3783,-0.0489,0.0000,0.0000,0.0000,0.0,0.0002,0.0175,0.0002,0.0002


In [7]:
T('robust_trend_test_size.csv')  # false-trend rate on trendless series (ideal 0.05)

,phi,n_years,reps,false_trend_rate_original,false_trend_rate_hamed_rao_all_lags,false_trend_rate_hamed_rao_lags_1_3,false_trend_rate_tfpw
0,0.0,32,1000,0.043,0.078,0.076,0.052
1,0.3,32,1000,0.141,0.150,0.136,0.146
2,0.5,32,1000,0.236,0.214,0.175,0.276
3,0.7,32,1000,0.345,0.256,0.200,0.443
4,0.8,32,1000,0.489,0.344,0.285,0.599
5,0.9,32,1000,0.618,0.449,0.374,0.739


Steady rise or one-off shift? Best single level-shift break against a linear trend (BIC), with the trend on each side of the break.

In [8]:
br = T('robust_trend_breaks.csv')
print(f"linear trend preferred in {(br.preferred_by_bic == 'linear trend').sum()} of {len(br)}; "
      f"rising on both sides in {((br.sen_slope_per_decade_before > 0) & (br.sen_slope_per_decade_after > 0)).sum()} of {len(br)}")
br.round(3)

linear trend preferred in 22 of 25; rising on both sides in 25 of 25


,spec,break_first_year_after,bic_step,bic_linear_trend,sen_slope_per_decade_before,sen_slope_per_decade_after,mk_p_before,mk_p_after,preferred_by_bic
0,Distance band 4 km | binary,2014,-201.513,-250.507,0.057,0.060,0.000,0.000,linear trend
1,Distance band 4 km | row-standardised,2009,-191.020,-231.636,0.102,0.057,0.000,0.000,linear trend
2,Distance band 6 km | binary,2015,-197.459,-258.918,0.063,0.084,0.000,0.000,linear trend
3,Distance band 6 km | row-standardised,2008,-212.648,-263.910,0.066,0.052,0.002,0.000,linear trend
4,Distance band 8 km | binary,2014,-232.031,-260.262,0.027,0.039,0.008,0.001,linear trend
5,Distance band 8 km | row-standardised,2014,-244.999,-261.493,0.018,0.024,0.005,0.044,linear trend
6,Inverse distance | raw,2014,-306.921,-312.628,0.001,0.014,0.575,0.006,linear trend
7,Inverse distance | row-standardised,2014,-305.442,-315.376,0.006,0.010,0.046,0.037,linear trend
8,kNN k=10 | binary,2002,-269.776,-267.920,0.002,0.012,1.000,0.000,step
9,kNN k=10 | row-standardised,2002,-269.776,-267.920,0.002,0.012,1.000,0.000,step


In [9]:
T('robust_moran_ci.csv').query('year in [1995, 2005, 2015, 2019, 2026]').round(3)

,year,I,lo95,hi95,p_perm
0,1995,0.450,0.443,0.461,0.002
10,2005,0.491,0.480,0.499,0.002
20,2015,0.553,0.543,0.562,0.002
24,2019,0.583,0.576,0.589,0.002
31,2026,0.604,0.590,0.616,0.002


In [10]:
T('robust_block_moran.csv').pivot(index='window', columns='band_km', values='I').round(3)

band_km,0.5,1.0,2.0
window,,,
1996-1998,0.754,0.648,0.564
1999-2001,0.756,0.673,0.595
2002-2004,0.760,0.696,0.623
2005-2007,0.743,0.674,0.605
2008-2010,0.742,0.670,0.599
2011-2013,0.728,0.650,0.574
2014-2016,0.753,0.668,0.592
2017-2019,0.788,0.689,0.613
2020-2022,0.775,0.676,0.604


## 4. Regional share of cross-town inequality and price dispersion

In [11]:
from data.loader import TOWN_COORDS
out = []
for y, g in t4[t4.town.isin(TOWN_COORDS)].groupby('year'):
    if y < 1995: continue
    ineq = S.inequality(g.price_ref.values, g.region.values)
    out.append((y, round(ineq['cv_pct'], 1), round(ineq['between_share_pct'], 0)))
pd.DataFrame(out, columns=['year', 'CV %', 'between-region share %']).set_index('year').loc[[1995, 2005, 2015, 2019, 2022, 2026]]

,CV %,between-region share %
year,,
1995,14.5,71.0
2005,13.7,73.0
2015,20.0,73.0
2019,23.5,78.0
2022,17.2,77.0
2026,18.9,78.0


## 5. Lease discount

In [12]:
T('hedonic_fit_summary.csv').set_index('year').lease_60_vs_90.loc[[2004, 2010, 2015, 2019, 2022, 2026]]

year
2004    0.8844
2010    0.8415
2015    0.8089
2019    0.7232
2022    0.7478
2026    0.7361
Name: lease_60_vs_90, dtype: float64

## 6. Rail access and the station event study (descriptive)

In [13]:
a = T('rail_access_town_year.csv'); a[a.town == 'All HDB blocks'].set_index('year')[['share_within_1km']].loc[[1990, 2000, 2010, 2020, 2026]].round(1)

,share_within_1km
year,
1990,55.8
2000,59.7
2010,76.4
2020,83.6
2026,84.9


In [14]:
T('rail_event_study.csv').round(2)

,k,effect_log,effect_pct,lo95_pct,hi95_pct,n_blocks
0,-6,-0.01,-1.20,-1.57,-0.80,1375
1,-5,-0.02,-1.56,-2.02,-1.08,1338
2,-4,-0.01,-0.71,-1.11,-0.30,1358
3,-3,-0.00,-0.47,-0.83,-0.11,1345
4,-2,-0.00,-0.25,-0.56,0.09,1344
5,-1,0.00,0.00,0.00,0.00,1339
6,0,0.01,0.88,0.53,1.21,1340
7,1,0.01,1.42,1.01,1.79,1322
8,2,0.01,1.18,0.78,1.56,1289
9,3,0.01,1.13,0.68,1.52,1312


## 7. Price model

In [15]:
ov = T('audit_backtest_overall.csv'); ov[ov['mode'] == 'forecast'][['model', 'mape_pct', 'within_10pct', 'r2']]

,model,mape_pct,within_10pct,r2
0,naive,12.02,54.2,0.7071
1,hedonic_ols,7.35,73.2,0.9133
2,rf_v1,9.38,62.9,0.8431
3,rf_indexed,7.87,71.3,0.8924
4,hgb,4.78,89.3,0.9630
6,hgb_te,4.31,92.5,0.9648
8,hgb_te_geo,4.18,93.1,0.9678
10,hgb_te_geo_temporal,4.10,93.5,0.9730


In [16]:
T('audit_horizons.csv')

,bucket,n,mape_pct,median_ape_pct,naive_mape_pct,bias_pct,coverage80_pct
0,0-5 months,179997,4.18,3.34,11.97,-1.60,80.19
1,6-11 months,181862,6.29,5.47,13.28,-4.66,59.78
2,12-17 months,171552,9.05,8.47,15.02,-8.05,40.16
3,18-23 months,162818,12.00,11.67,17.07,-11.45,25.49


In [17]:
T('market_forecast_baselines.csv')

,method,h,mae_pct,rmse_pct,p90_abs_pct,n
0,12-month trend,6,2.54,3.35,5.48,300
1,12-month trend,12,5.59,7.12,11.70,300
2,12-month trend,24,12.38,15.45,23.63,296
3,36-month trend,6,2.66,3.69,6.85,300
4,36-month trend,12,5.07,7.03,13.05,300
5,36-month trend,24,10.43,14.15,24.99,296
6,Random walk (no change),6,3.59,4.51,7.01,300
7,Random walk (no change),12,6.64,8.14,12.64,300
8,Random walk (no change),24,11.57,14.12,22.62,296
